# Forecasting Time-series
###### Juan Camilo Mercado

## Objetivo del negocio

El objetivo de este proyecto es desarrollar un sistema de forecasting de demanda capaz de predecir las unidades vendidas de diferentes productos y tiendas utilizando variables históricas, operativas y contextuales. El modelo busca apoyar decisiones relacionadas con inventario, precios, promociones y planeación operativa, reduciendo riesgos de desabastecimiento o exceso de stock.

Además, el proyecto pretende identificar cómo factores como precios, descuentos, estacionalidad, clima y comportamiento histórico afectan la demanda, permitiendo una toma de decisiones más estratégica y basada en datos.

---

## Descripción del caso

El caso corresponde a un entorno retail con múltiples tiendas y productos distribuidos en diferentes regiones. El dataset contiene información histórica de ventas, inventario, precios, promociones y variables externas que influyen en la demanda.

Para abordar el problema, se implementa un modelo de forecasting basado en XGBoost utilizando ingeniería de variables temporales como lags y promedios móviles, permitiendo capturar patrones históricos y relaciones no lineales en las ventas.

Adicionalmente, se desarrolla una aplicación interactiva en Streamlit para visualizar predicciones y simular diferentes escenarios de negocio.

---

## Preguntas de negocio

* ¿Cómo evolucionará la demanda futura por producto y tienda en los siguientes días?
* ¿Qué variables impactan más las ventas?
* ¿Cómo afectan el precio y los descuentos la demanda?
* ¿Existen niveles óptimos de precio o descuento?
* ¿Cómo ajustar inventarios según el forecast?


## Carga de Librerías

In [21]:
# Manipulación de datos
import polars as pl
import pandas as pd
import numpy as np

# Modelado
from xgboost import XGBRegressor

#Visualización
import plotly.graph_objects as go

# Validación y parametrización
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller
from plotly.subplots import make_subplots

# Guardar y cargar modelos
import joblib

### Carga de datos

In [22]:
df = pl.read_csv('./Data/retail_store_inventory.csv')

### Entendimiento de datos

In [23]:
df.shape

(73100, 15)

In [24]:
df.describe()

statistic,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,f64,f64,str
"""count""","""73100""","""73100""","""73100""","""73100""","""73100""",73100.0,73100.0,73100.0,73100.0,73100.0,73100.0,"""73100""",73100.0,73100.0,"""73100"""
"""null_count""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,"""0""",0.0,0.0,"""0"""
"""mean""",null,null,null,null,null,274.469877,136.46487,110.004473,141.49472,55.135108,10.009508,null,0.497305,55.146077,null
"""std""",null,null,null,null,null,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,null,0.499996,26.191408,null
"""min""","""2022-01-01""","""S001""","""P0001""","""Clothing""","""East""",50.0,0.0,20.0,-9.99,10.0,0.0,"""Cloudy""",0.0,5.03,"""Autumn"""
"""25%""",null,null,null,null,null,162.0,49.0,65.0,53.67,32.65,5.0,null,0.0,32.68,null
"""50%""",null,null,null,null,null,273.0,107.0,110.0,113.02,55.05,10.0,null,0.0,55.01,null
"""75%""",null,null,null,null,null,387.0,203.0,155.0,208.05,77.86,15.0,null,1.0,77.82,null
"""max""","""2024-01-01""","""S005""","""P0020""","""Toys""","""West""",500.0,499.0,200.0,518.55,100.0,20.0,"""Sunny""",1.0,104.94,"""Winter"""


Se observa que en calidad de datos no hay valores faltantes ni de tipos incorrecos, por lo que se infiere cumple con las reglas del negocio además de que todas las columnas tienen el tipo de dato correcto. No hay duplicidad en os datos.

In [25]:
numeric_cols = [
    col for col, dtype in zip(df.columns, df.dtypes)
    if dtype in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64
    )
]

outlier_summary = []
for col in numeric_cols:
    q1 = df.select(pl.col(col).quantile(0.25)).to_numpy()[0][0]
    q3 = df.select(pl.col(col).quantile(0.75)).to_numpy()[0][0]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = df.filter((pl.col(col) < lower) | (pl.col(col) > upper)).height
    outlier_summary.append((col, q1, q3, lower, upper, count))

outliers_df = pl.DataFrame(
    outlier_summary,
    schema=["column", "q1", "q3", "lower", "upper", "outlier_count"]
)

print(outliers_df)

if "Units Sold" in numeric_cols:
    units_limits = outliers_df.filter(pl.col("column") == "Units Sold").select(["lower", "upper"]).to_dicts()[0]
    print("\nEjemplos de outliers en Units Sold:")
    print(
        df.filter(
            (pl.col("Units Sold") < units_limits["lower"])
            | (pl.col("Units Sold") > units_limits["upper"])
        )
        .select(["Date", "Store ID", "Product ID", "Units Sold"])
        .head(10)
        .to_pandas()
    )

shape: (8, 6)
┌────────────────────┬───────┬────────┬─────────┬─────────┬───────────────┐
│ column             ┆ q1    ┆ q3     ┆ lower   ┆ upper   ┆ outlier_count │
│ ---                ┆ ---   ┆ ---    ┆ ---     ┆ ---     ┆ ---           │
│ str                ┆ f64   ┆ f64    ┆ f64     ┆ f64     ┆ i64           │
╞════════════════════╪═══════╪════════╪═════════╪═════════╪═══════════════╡
│ Inventory Level    ┆ 162.0 ┆ 387.0  ┆ -175.5  ┆ 724.5   ┆ 0             │
│ Units Sold         ┆ 49.0  ┆ 203.0  ┆ -182.0  ┆ 434.0   ┆ 715           │
│ Units Ordered      ┆ 65.0  ┆ 155.0  ┆ -70.0   ┆ 290.0   ┆ 0             │
│ Demand Forecast    ┆ 53.67 ┆ 208.05 ┆ -177.9  ┆ 439.62  ┆ 732           │
│ Price              ┆ 32.65 ┆ 77.86  ┆ -35.165 ┆ 145.675 ┆ 0             │
│ Discount           ┆ 5.0   ┆ 15.0   ┆ -10.0   ┆ 30.0    ┆ 0             │
│ Holiday/Promotion  ┆ 0.0   ┆ 1.0    ┆ -1.5    ┆ 2.5     ┆ 0             │
│ Competitor Pricing ┆ 32.68 ┆ 77.82  ┆ -35.03  ┆ 145.53  ┆ 0             

C:\Users\CAMILO\AppData\Local\Temp\ipykernel_49708\3985465924.py:20: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  outliers_df = pl.DataFrame(


Se observa que los outliers usando RIQ1.5 mantienen una tendencia y no se extrapolan a comportamientos diferentes, por o que se considera picos en las ventas objeto importante de representar en ventas, por lo que no requieren mayor tratamiento.

In [26]:
fig_init = go.Figure()

fig_init.add_trace(go.Scatter(
    x=df['Date'],
    y=df['Units Sold'],
    mode='lines',
    name='Units Sold',
    line=dict(color='blue', width=2)
))

fig_init.show()

In [27]:
sales_agg = (
    df.group_by("Date")
      .agg(pl.col("Units Sold").sum().alias("Units Sold"))
      .sort("Date")
      .to_pandas()
)
sales_agg["Date"] = pd.to_datetime(sales_agg["Date"])
sales_ts = sales_agg.set_index("Date").asfreq("D").fillna(0)["Units Sold"]

decomp = seasonal_decompose(sales_ts, model="additive", period=7)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=["Serie original", "Tendencia", "Estacionalidad", "Residuo"]
)

fig.add_trace(go.Scatter(x=sales_ts.index, y=sales_ts, name="Original"), row=1, col=1)
fig.add_trace(go.Scatter(x=sales_ts.index, y=decomp.trend, name="Tendencia"), row=2, col=1)
fig.add_trace(go.Scatter(x=sales_ts.index, y=decomp.seasonal, name="Estacionalidad"), row=3, col=1)
fig.add_trace(go.Scatter(x=sales_ts.index, y=decomp.resid, name="Residuo"), row=4, col=1)

fig.update_layout(height=900, title_text="Descomposición de la serie agregada de unidades vendidas")
fig.show()

lag_acf = acf(sales_ts, nlags=30, fft=True)

fig_acf = go.Figure(go.Bar(x=list(range(len(lag_acf))), y=lag_acf))
fig_acf.update_layout(
    title="Autocorrelación (ACF) de unidades vendidas",
    xaxis_title="Rezago (lag)",
    yaxis_title="ACF"
)
fig_acf.show()

adf_result = adfuller(sales_ts)
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])
print("Lags used:", adf_result[2])
print("Number of observations:", adf_result[3])
for key, value in adf_result[4].items():
    print(f"Critical Value ({key}): {value}")
print("ACF lag 7:", lag_acf[7])

ADF Statistic: -26.379506701264116
p-value: 0.0
Lags used: 0
Number of observations: 730
Critical Value (1%): -3.4393396487377155
Critical Value (5%): -2.865507363200066
Critical Value (10%): -2.5688826684180897
ACF lag 7: -0.009755512228036209


En la componente de tendencia se observa que las ventas mantienen un comportamiento relativamente estable a lo largo del tiempo, oscilando aproximadamente entre 13k y 14.5k unidades. No se identifica una tendencia creciente o decreciente sostenida, aunque sí existen fluctuaciones moderadas durante distintos períodos. Esto sugiere que la demanda general del negocio se mantiene estable en el largo plazo, sin cambios estructurales drásticos en el volumen total de ventas.

La componente de estacionalidad muestra un patrón repetitivo y consistente en el tiempo. Esto indica que existen comportamientos periódicos en la demanda que se repiten de manera regular, posiblemente asociados a ciclos semanales, promociones recurrentes o patrones de consumo propios del negocio retail. La presencia de estacionalidad confirma que ciertos momentos del tiempo presentan comportamientos previsibles de demanda.

Por otro lado, la componente residual representa la variabilidad no explicada por la tendencia ni la estacionalidad. Aunque los residuos se concentran alrededor de cero, se observan fluctuaciones importantes y algunos picos pronunciados positivos y negativos. Esto indica que todavía existen factores externos o eventos aleatorios que afectan las ventas y que no son completamente capturados por los componentes temporales básicos.

---

La gráfica de autocorrelación (ACF) y la prueba ADF permiten analizar si la serie temporal de unidades vendidas presenta dependencia temporal y estacionariedad, dos aspectos fundamentales en problemas de forecasting.

En la gráfica ACF se observa que, aparte del rezago 0, la mayoría de los rezagos presentan valores cercanos a cero. Esto indica que no existe una dependencia temporal fuerte entre las observaciones pasadas y futuras de la serie. En otras palabras, las ventas no muestran una autocorrelación lineal significativa en la mayoría de los rezagos analizados.

Sin embargo, se identifican pequeños picos positivos alrededor de ciertos rezagos, especialmente cerca del lag 10 y algunos rezagos entre 18 y 21. Esto podría sugerir patrones débiles de comportamiento periódico o pequeñas señales de estacionalidad, aunque no suficientemente fuertes como para indicar una estructura temporal dominante.

Por otro lado, la prueba Dickey-Fuller aumentada (ADF) muestra un estadístico de -26.38 con un p-value de 0.0, muy por debajo del umbral típico de significancia de 0.05. Esto permite rechazar la hipótesis nula de no estacionariedad, concluyendo que la serie es estacionaria. Es decir, las propiedades estadísticas de la serie, como la media y la varianza, permanecen relativamente estables a lo largo del tiempo.

Este resultado es importante porque indica que la serie no requiere diferenciación adicional para estabilizarla, lo que facilita el modelado y reduce la complejidad temporal del problema.

### Feature Engineering

En esta sección se crean variables derivadas del tiempo y se transforman variables categóricas. El día de la semana y el mes permiten capturar estacionalidad en la demanda, como patrones de consumo semanales o mensuales. Estas variables permiten al modelo incorporar contexto de negocio además de información temporal.

Se calculan variables que aportan valor como las ganancias titales en la fecha, ganancias netas (sin descuento) y la referencia con el precio de la competencia.

In [28]:
df = df.with_columns([
    pl.col("Date").str.strptime(pl.Date, '%Y-%m-%d').dt.weekday().alias("weekday"),
    pl.col("Date").str.strptime(pl.Date, '%Y-%m-%d').dt.month().alias("month"),
])


In [29]:
df = df.with_columns([(df["Units Sold"] * df["Price"]).alias("Revenue")])

In [30]:
df = df.with_columns([(df["Revenue"] - df["Discount"]).alias("Net Revenue"), 
                      
                     (df["Price"] - df["Competitor Pricing"]).alias("Competitor Price Difference")])

Igualmente, se construyen las variables lags, que representan valores pasados de la variable objetivo, lo que permite capturar dependencia temporal. El lag de un día refleja inercia inmediata en la demanda, mientras que el lag de siete días captura patrones semanales. La media móvil de siete días suaviza la serie y ayuda a identificar tendencias generales. Todas estas transformaciones se realizan por combinación de tienda y producto para preservar la independencia de cada serie temporal.

In [31]:
df = df.with_columns([
    pl.col("Units Sold")
      .shift(1)
      .over(["Store ID", "Product ID"])
      .alias("lag_1"),

    pl.col("Units Sold")
      .shift(7)
      .over(["Store ID", "Product ID"])
      .alias("lag_7"),
])

df = df.with_columns([
    pl.col("Units Sold")
      .rolling_mean(window_size=7)
      .over(["Store ID", "Product ID"])
      .alias("rolling_mean_7")
])

df.tail()

Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,weekday,month,Revenue,Net Revenue,Competitor Price Difference,lag_1,lag_7,rolling_mean_7
str,str,str,str,str,i64,i64,i64,f64,f64,i64,str,i64,f64,str,i8,i8,f64,f64,f64,i64,i64,f64
"""2024-01-01""","""S005""","""P0016""","""Furniture""","""East""",96,8,127,18.46,73.73,20,"""Snowy""",0,72.45,"""Winter""",1,1,589.84,569.84,1.28,54,43,82.571429
"""2024-01-01""","""S005""","""P0017""","""Toys""","""North""",313,51,101,48.43,82.57,10,"""Cloudy""",0,83.78,"""Autumn""",1,1,4211.07,4201.07,-1.21,162,119,125.285714
"""2024-01-01""","""S005""","""P0018""","""Clothing""","""West""",278,36,151,39.65,11.11,10,"""Rainy""",0,10.91,"""Winter""",1,1,399.96,389.96,0.2,111,5,99.857143
"""2024-01-01""","""S005""","""P0019""","""Toys""","""East""",374,264,21,270.52,53.14,20,"""Rainy""",0,55.8,"""Spring""",1,1,14028.96,14008.96,-2.66,17,213,144.428571
"""2024-01-01""","""S005""","""P0020""","""Groceries""","""East""",117,6,165,2.33,78.39,20,"""Rainy""",1,79.52,"""Spring""",1,1,470.34,450.34,-1.13,40,31,115.714286


### Preparación de datos

In [32]:
df = df.sort(["Store ID", "Product ID", "Date"])

In [33]:
pdf = df.to_pandas()

In [34]:
target = "Units Sold"

A todas las variables categóricas se codifican para poder usarlas en las predicciones.

In [35]:
pdf = pd.get_dummies(pdf, columns=[
    "Store ID",
    "Product ID",
    "Weather Condition",
    "Holiday/Promotion",
    "Seasonality",
    "Category",
    "Region"
])

### Entrenamiento de modelos

In [36]:
split = int(len(pdf) * 0.8)

train = pdf[:split]
test = pdf[split:]

In [37]:
X_train = train.drop(columns=[target, "Date"])
y_train = train[target]

X_test = test.drop(columns=[target, "Date"])
y_test = test[target]

Se establecen las condiciones para hacer GridSearch de hiperparámetros más adelante.

In [38]:
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {

    "n_estimators": [100, 300],

    "learning_rate": [0.01, 0.05],

    "max_depth": [4, 6],

    "subsample": [0.8, 1.0],

    "colsample_bytree": [0.8, 1.0]
}



Se elige el modelo XGBoost, que es capaz de capturar relaciones no lineales y complejas entre variables habiendo muchas categorías, al contrario que (S)ARIMA o Prophet. Los hiperparámetros controlan la complejidad del modelo y ayudan a evitar overfitting, permitiendo una buena generalización sobre datos no vistos.

In [39]:
xgb = XGBRegressor(
    random_state=777
)

Se buscan los mejores hiperparámetros con GridSearch que ayuda a probar todas las combinaciones dadas eligiendo la de mejor ajuste.

In [40]:
grid_search = GridSearchCV(

    estimator=xgb,

    param_grid=param_grid,

    scoring="neg_mean_absolute_error",

    cv=tscv,

    verbose=1,

    n_jobs=-1
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

Fitting 3 folds for each of 32 candidates, totalling 96 fits


Con el mejor modelo, se entrena al mismo.

In [41]:

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

En esta etapa se generan predicciones sobre el conjunto de prueba. Estas predicciones representan la estimación de la demanda futura basada en los patrones aprendidos durante el entrenamiento.

In [42]:
preds = model.predict(X_test)

En esta celda se evalúa el desempeño del modelo utilizando métricas estándar de regresión. El MAE mide el error absoluto promedio, el RMSE penaliza errores grandes y el MAPE expresa el error en términos porcentuales, lo cual es especialmente útil para interpretación en contextos de negocio.

In [43]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return mae, rmse


evaluate(y_test, preds)

(2.838449001312256, np.float64(3.9572580315845007))

El modelo obtuvo un MAE de 2.83, lo que indica que, en promedio, las predicciones de demanda presentan un error aproximado de 2.83 unidades vendidas respecto a los valores reales. Esta métrica permite interpretar el desempeño del modelo de manera directa en las mismas unidades del negocio, facilitando la comprensión del impacto práctico del error. 

Por otro lado, el RMSE obtenido fue de 3.96, valor ligeramente superior al MAE debido a que esta métrica penaliza con mayor intensidad los errores grandes. 

La diferencia moderada entre ambas métricas sugiere que el modelo mantiene un comportamiento relativamente estable y que no presenta una cantidad significativa de predicciones extremadamente alejadas de los valores reales. 

En conjunto, estos resultados indican que el modelo logra capturar adecuadamente los patrones de demanda presentes en los datos, manteniendo un nivel de error razonable para un problema de forecasting en retail.

In [44]:
results_df = test[['Date']].copy()
results_df['Real'] = y_test.values
results_df['Predicción'] = preds

results_df['Date'] = pd.to_datetime(results_df['Date'])


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results_df['Date'],
    y=results_df['Real'],
    mode='lines',
    name='Valores Reales',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=results_df['Date'],
    y=results_df['Predicción'],
    mode='lines',
    name='Predicciones',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title='Predicciones vs Valores Reales (Units Sold)',
    xaxis_title='Fecha',
    yaxis_title='Units Sold',
    hovermode='x unified',
    height=600
)

fig.show()

La gráfica muestra que predice con cercana precisión, pues visualmente se observa que el error no es relativamente grande comparado al valor real, por lo que se considera como confiable.

Teniendo este modelo analizado y funcionando correctamente sin caer en el overfit ni sin ser muy impreciso se decide exportarlo para su uso y sus columnas para la construcción del reporte.

In [45]:
joblib.dump(model, "./Models/xgboost_forecast.pkl")

model_columns = X_train.columns
joblib.dump(model_columns, "./Models/model_columns.pkl")

['./Models/model_columns.pkl']